# 03 — NLP Experiment: Incident-title themes

This notebook explores title text using TF-IDF and K-Means. Clusters are exploratory groupings, not verified hazard categories. It does not assume `hazard` IDs are human-readable labels.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.cluster import KMeans

ROOT = Path.cwd()
if not (ROOT / 'data').exists() and (ROOT.parent / 'data').exists(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from src.data_utils import load_dataset

df = load_dataset()
text_df = df.loc[df['title'].notna(), ['id_incident', 'title', 'incidentOn', 'hazard']].copy()
text_df['title'] = text_df['title'].astype(str).str.strip()
text_df = text_df[text_df['title'].ne('')].drop_duplicates(subset=['id_incident'])
print('Titles available:', len(text_df))
text_df.head()

## 1. TF-IDF representation

In [ ]:
vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words='english',
    ngram_range=(1, 2),
    min_df=5,
    max_features=3000,
    sublinear_tf=True,
)
X_text = vectorizer.fit_transform(text_df['title'])
print('TF-IDF matrix shape:', X_text.shape)

## 2. K-Means clusters and top terms

In [ ]:
N_CLUSTERS = 6
km = KMeans(n_clusters=N_CLUSTERS, random_state=42, n_init=10)
text_df['title_cluster'] = km.fit_predict(X_text)
terms = vectorizer.get_feature_names_out()
for cluster_id, center in enumerate(km.cluster_centers_):
    top_idx = center.argsort()[::-1][:12]
    print(f'\nCluster {cluster_id} — top terms: ' + ', '.join(terms[top_idx]))
    display(text_df.loc[text_df['title_cluster'].eq(cluster_id), 'title'].head(8).to_frame())
display(text_df['title_cluster'].value_counts().sort_index().rename_axis('cluster').to_frame('records'))

## 3. Export cluster assignments

In [ ]:
out_cols = ['id_incident', 'title', 'incidentOn', 'hazard', 'title_cluster']
text_df[out_cols].to_csv(ROOT / 'reports' / 'nlp_title_clusters.csv', index=False)
print('Saved reports/nlp_title_clusters.csv')

## Interpretation cautions
- Inspect example titles and top terms before naming a cluster.
- Titles may contain English, Nepali, transliteration, and administrative names; English stop-word filtering is not full multilingual NLP.
- Clustering does not prove that groups are distinct disaster types.
- A more rigorous next step is to use an authoritative hazard-label mapping or manually validated labels and evaluate classification on held-out data.